# EDA and Visual Checks for CT Needle Segmentation

This notebook audits the dataset before model development. It focuses on file alignment, image/mask integrity, needle area statistics, grayscale/RGB checks, intensity behavior, and qualitative overlays.

Key dataset assumptions to verify:

- `trainImages` contains the training CT scans.
- `trainMasks` contains the ground-truth segmentation masks.
- `testImages` contains the testing CT scans.
- Training images and masks should be paired by `imageID`, even if their raw filenames are not identical.
- Needle pixels are positive mask pixels; background pixels are zero.

> **Important:** This notebook intentionally aligns files by parsed image ID rather than filename equality. For example, `1234.jpg` should align with `1234_mask.png`.

## 1. Imports and Project Paths

In [ ]:
from pathlib import Path
import random
import re
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

try:
    import cv2
    HAS_CV2 = True
except ImportError:
    HAS_CV2 = False
    print("OpenCV is not installed. Connected-component checks will be skipped unless cv2 is installed.")

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["image.cmap"] = "gray"
random.seed(42)
np.random.seed(42)

PROJECT_ROOT = Path.cwd()

TRAIN_IMAGES_ROOT = PROJECT_ROOT / "trainImages"
TRAIN_MASKS_ROOT = PROJECT_ROOT / "trainMasks"
TEST_IMAGES_ROOT = PROJECT_ROOT / "testImages"
TRAIN_CSV = PROJECT_ROOT / "trainSet.csv"

PROJECT_ROOT

## 2. Locate Image Files Robustly

The local project currently uses nested directories such as `trainImages/trainImages`. The helper below searches recursively and accepts common image extensions so the notebook works whether the files are `.jpg`, `.jpeg`, or `.png`.

In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}


def find_image_files(root):
    root = Path(root)
    files = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
    return sorted(files)


train_image_paths = find_image_files(TRAIN_IMAGES_ROOT)
train_mask_paths = find_image_files(TRAIN_MASKS_ROOT)
test_image_paths = find_image_files(TEST_IMAGES_ROOT)

print(f"Train images: {len(train_image_paths)}")
print(f"Train masks:  {len(train_mask_paths)}")
print(f"Test images:  {len(test_image_paths)}")

print("\nSample train image paths:")
for p in train_image_paths[:5]:
    print(" ", p.relative_to(PROJECT_ROOT))

print("\nSample train mask paths:")
for p in train_mask_paths[:5]:
    print(" ", p.relative_to(PROJECT_ROOT))

## 3. Extension, Mode, and Shape Audit

This section checks whether the dataset is actually PNG/JPG, grayscale/RGB, and consistently sized.

In [ ]:
def image_file_summary(paths, label):
    rows = []
    for path in paths:
        with Image.open(path) as img:
            rows.append(
                {
                    "label": label,
                    "path": str(path.relative_to(PROJECT_ROOT)),
                    "suffix": path.suffix.lower(),
                    "mode": img.mode,
                    "width": img.size[0],
                    "height": img.size[1],
                }
            )
    return pd.DataFrame(rows)


train_image_summary = image_file_summary(train_image_paths, "train_image")
train_mask_summary = image_file_summary(train_mask_paths, "train_mask")
test_image_summary = image_file_summary(test_image_paths, "test_image")

file_summary = pd.concat([train_image_summary, train_mask_summary, test_image_summary], ignore_index=True)

display(file_summary.groupby(["label", "suffix"]).size().rename("count").reset_index())
display(file_summary.groupby(["label", "mode"]).size().rename("count").reset_index())
display(file_summary.groupby(["label", "width", "height"]).size().rename("count").reset_index())

Expected checks:

- Training images, masks, and test images should all be `512 x 512`.
- CT images are expected to be grayscale.
- Masks should be grayscale or otherwise convertible to grayscale.
- If image folders contain `.jpg` instead of `.png`, downstream loading code should not hardcode `.png`.

## 4. Parse Image IDs and Check Train Image / Mask Alignment

Raw filenames are not expected to match exactly because masks often contain a suffix like `_mask`. The correct alignment key is the numeric image ID.

In [ ]:
def parse_train_image_id(path):
    return path.stem


def parse_train_mask_id(path):
    stem = path.stem
    if stem.endswith("_mask"):
        return stem[:-5]
    return stem


def parse_test_image_id(path):
    return path.stem


train_image_by_id = {parse_train_image_id(p): p for p in train_image_paths}
train_mask_by_id = {parse_train_mask_id(p): p for p in train_mask_paths}
test_image_by_id = {parse_test_image_id(p): p for p in test_image_paths}

train_image_ids = set(train_image_by_id)
train_mask_ids = set(train_mask_by_id)

missing_masks = sorted(train_image_ids - train_mask_ids, key=lambda x: int(x) if x.isdigit() else x)
missing_images = sorted(train_mask_ids - train_image_ids, key=lambda x: int(x) if x.isdigit() else x)
aligned_ids = sorted(train_image_ids & train_mask_ids, key=lambda x: int(x) if x.isdigit() else x)

print(f"Training image IDs: {len(train_image_ids)}")
print(f"Training mask IDs:  {len(train_mask_ids)}")
print(f"Aligned pairs:      {len(aligned_ids)}")
print(f"Missing masks:      {len(missing_masks)}")
print(f"Missing images:     {len(missing_images)}")

if missing_masks:
    print("\nImage IDs missing masks:", missing_masks[:20])
if missing_images:
    print("\nMask IDs missing images:", missing_images[:20])

In [ ]:
alignment_df = pd.DataFrame(
    {
        "imageID": aligned_ids,
        "image_path": [str(train_image_by_id[i].relative_to(PROJECT_ROOT)) for i in aligned_ids],
        "mask_path": [str(train_mask_by_id[i].relative_to(PROJECT_ROOT)) for i in aligned_ids],
    }
)

display(alignment_df.head())
display(alignment_df.tail())

## 5. Compare Folder Alignment Against `trainSet.csv`

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
train_df["imageID"] = train_df["imageID"].astype(str)

csv_ids = set(train_df["imageID"])

print(f"Rows in trainSet.csv: {len(train_df)}")
print(f"Unique CSV image IDs: {len(csv_ids)}")
print(f"CSV IDs missing train image files: {len(sorted(csv_ids - train_image_ids))}")
print(f"CSV IDs missing train mask files:  {len(sorted(csv_ids - train_mask_ids))}")
print(f"Train image IDs not in CSV:        {len(sorted(train_image_ids - csv_ids))}")
print(f"Train mask IDs not in CSV:         {len(sorted(train_mask_ids - csv_ids))}")

display(train_df["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="count"))
display(train_df.head())

Interpretation notes:

- `status == 1` should correspond to images with needle masks.
- `status == 0` should correspond to empty masks.
- If folder masks disagree with CSV status, trust the actual mask pixels during segmentation training and inspect the discrepancy.

## 6. Image and Mask Loading Helpers

In [ ]:
def load_grayscale_array(path):
    with Image.open(path) as img:
        return np.array(img.convert("L"))


def load_binary_mask(path):
    arr = load_grayscale_array(path)
    return (arr > 0).astype(np.uint8)


def image_id_to_paths(image_id):
    image_id = str(image_id)
    return train_image_by_id[image_id], train_mask_by_id[image_id]


sample_id = aligned_ids[0]
sample_image_path, sample_mask_path = image_id_to_paths(sample_id)
sample_image = load_grayscale_array(sample_image_path)
sample_mask = load_binary_mask(sample_mask_path)

print(sample_id, sample_image.shape, sample_image.dtype, sample_mask.shape, sample_mask.dtype)
print("Image min/max:", sample_image.min(), sample_image.max())
print("Mask unique values:", np.unique(sample_mask))

## 7. Mask Area and Needle Sparsity

Compute needle area as both raw positive pixels and percentage of the full image.

In [ ]:
mask_rows = []

for image_id in aligned_ids:
    mask = load_binary_mask(train_mask_by_id[image_id])
    positive_pixels = int(mask.sum())
    total_pixels = int(mask.size)
    mask_rows.append(
        {
            "imageID": image_id,
            "positive_pixels": positive_pixels,
            "total_pixels": total_pixels,
            "needle_area_pct": 100.0 * positive_pixels / total_pixels,
            "is_empty_mask": positive_pixels == 0,
        }
    )

mask_stats_df = pd.DataFrame(mask_rows)
mask_stats_df = mask_stats_df.merge(train_df[["imageID", "status"]], on="imageID", how="left")

display(mask_stats_df.describe(include="all"))
display(mask_stats_df.groupby(["status", "is_empty_mask"]).size().rename("count").reset_index())
display(mask_stats_df.sort_values("positive_pixels", ascending=False).head(10))
display(mask_stats_df.sort_values("positive_pixels", ascending=True).head(10))

In [ ]:
positive_area = mask_stats_df.loc[mask_stats_df["positive_pixels"] > 0, "needle_area_pct"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(mask_stats_df["positive_pixels"], bins=40, color="steelblue", edgecolor="black")
axes[0].set_title("Mask Positive Pixel Count")
axes[0].set_xlabel("Positive pixels")
axes[0].set_ylabel("Image count")

axes[1].hist(positive_area, bins=40, color="darkorange", edgecolor="black")
axes[1].set_title("Needle Area Percentage, Positive Masks Only")
axes[1].set_xlabel("Needle area (% of image)")
axes[1].set_ylabel("Image count")

plt.tight_layout()
plt.show()

## 8. CT Intensity Feature Exploration

This section summarizes image intensity distributions globally and compares needle pixels against background pixels for positive masks.

In [ ]:
intensity_rows = []

for image_id in aligned_ids:
    image = load_grayscale_array(train_image_by_id[image_id]).astype(np.float32)
    mask = load_binary_mask(train_mask_by_id[image_id]).astype(bool)

    row = {
        "imageID": image_id,
        "image_min": float(image.min()),
        "image_max": float(image.max()),
        "image_mean": float(image.mean()),
        "image_std": float(image.std()),
        "image_p01": float(np.percentile(image, 1)),
        "image_p50": float(np.percentile(image, 50)),
        "image_p99": float(np.percentile(image, 99)),
        "positive_pixels": int(mask.sum()),
    }

    if mask.sum() > 0:
        needle_values = image[mask]
        background_values = image[~mask]
        row.update(
            {
                "needle_mean": float(needle_values.mean()),
                "needle_std": float(needle_values.std()),
                "needle_p10": float(np.percentile(needle_values, 10)),
                "needle_p50": float(np.percentile(needle_values, 50)),
                "needle_p90": float(np.percentile(needle_values, 90)),
                "background_mean": float(background_values.mean()),
                "background_std": float(background_values.std()),
                "needle_minus_background_mean": float(needle_values.mean() - background_values.mean()),
            }
        )
    else:
        row.update(
            {
                "needle_mean": np.nan,
                "needle_std": np.nan,
                "needle_p10": np.nan,
                "needle_p50": np.nan,
                "needle_p90": np.nan,
                "background_mean": float(image.mean()),
                "background_std": float(image.std()),
                "needle_minus_background_mean": np.nan,
            }
        )

    intensity_rows.append(row)

intensity_df = pd.DataFrame(intensity_rows).merge(train_df[["imageID", "status"]], on="imageID", how="left")
display(intensity_df.describe())
display(intensity_df.loc[intensity_df["positive_pixels"] > 0, ["needle_mean", "background_mean", "needle_minus_background_mean"]].describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(intensity_df["image_mean"], bins=40, color="slateblue", edgecolor="black")
axes[0].set_title("Image Mean Intensity")

axes[1].hist(intensity_df["image_std"], bins=40, color="seagreen", edgecolor="black")
axes[1].set_title("Image Intensity Std")

delta = intensity_df["needle_minus_background_mean"].dropna()
axes[2].hist(delta, bins=40, color="crimson", edgecolor="black")
axes[2].axvline(0, color="black", linestyle="--", linewidth=1)
axes[2].set_title("Needle Mean - Background Mean")

plt.tight_layout()
plt.show()

Interpretation questions:

- Is the needle consistently brighter than the background?
- Are there positive masks where `needle_minus_background_mean` is near or below zero?
- Do test images have similar global intensity ranges to train images?

## 9. Train vs Test Image Feature Comparison

In [ ]:
def global_image_features(paths, label):
    rows = []
    for path in paths:
        image = load_grayscale_array(path).astype(np.float32)
        rows.append(
            {
                "split": label,
                "imageID": path.stem,
                "mean": float(image.mean()),
                "std": float(image.std()),
                "min": float(image.min()),
                "max": float(image.max()),
                "p01": float(np.percentile(image, 1)),
                "p50": float(np.percentile(image, 50)),
                "p99": float(np.percentile(image, 99)),
            }
        )
    return pd.DataFrame(rows)


train_features_df = global_image_features(train_image_paths, "train")
test_features_df = global_image_features(test_image_paths, "test")
split_features_df = pd.concat([train_features_df, test_features_df], ignore_index=True)

display(split_features_df.groupby("split")[["mean", "std", "min", "max", "p01", "p50", "p99"]].describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for split, group in split_features_df.groupby("split"):
    axes[0].hist(group["mean"], bins=30, alpha=0.5, label=split)
    axes[1].hist(group["std"], bins=30, alpha=0.5, label=split)
    axes[2].hist(group["p99"], bins=30, alpha=0.5, label=split)

axes[0].set_title("Train vs Test Mean Intensity")
axes[1].set_title("Train vs Test Intensity Std")
axes[2].set_title("Train vs Test 99th Percentile")

for ax in axes:
    ax.legend()

plt.tight_layout()
plt.show()

## 10. Connected-Component Checks for Masks

This checks whether each positive mask is one connected component or whether labels are fragmented. It requires OpenCV.

In [ ]:
component_rows = []

if HAS_CV2:
    for image_id in aligned_ids:
        mask = load_binary_mask(train_mask_by_id[image_id])
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
        component_areas = [int(stats[label, cv2.CC_STAT_AREA]) for label in range(1, num_labels)]
        component_rows.append(
            {
                "imageID": image_id,
                "component_count": len(component_areas),
                "largest_component_area": max(component_areas) if component_areas else 0,
                "total_positive_area": int(mask.sum()),
            }
        )

    components_df = pd.DataFrame(component_rows).merge(train_df[["imageID", "status"]], on="imageID", how="left")
    display(components_df.describe())
    display(components_df["component_count"].value_counts().sort_index().rename_axis("component_count").reset_index(name="image_count"))
    display(components_df.sort_values("component_count", ascending=False).head(10))
else:
    components_df = pd.DataFrame()
    print("Skipping component checks because cv2 is unavailable.")

## 11. Visualization Helpers

In [ ]:
def make_overlay(image, mask, color=(255, 0, 0), alpha=0.45):
    image_norm = image.astype(np.float32)
    if image_norm.max() > image_norm.min():
        image_norm = (image_norm - image_norm.min()) / (image_norm.max() - image_norm.min())
    image_rgb = np.stack([image_norm, image_norm, image_norm], axis=-1)

    color_arr = np.array(color, dtype=np.float32) / 255.0
    overlay = image_rgb.copy()
    overlay[mask.astype(bool)] = (1 - alpha) * overlay[mask.astype(bool)] + alpha * color_arr
    return np.clip(overlay, 0, 1)


def plot_image_mask_overlay(image_id):
    image_path, mask_path = image_id_to_paths(image_id)
    image = load_grayscale_array(image_path)
    mask = load_binary_mask(mask_path)
    overlay = make_overlay(image, mask)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(image, cmap="gray")
    axes[0].set_title(f"Image {image_id}")
    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title(f"Mask | pixels={int(mask.sum())}")
    axes[2].imshow(overlay)
    axes[2].set_title("Overlay")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


def plot_overlay_grid(image_ids, title):
    n = len(image_ids)
    cols = 4
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, image_id in zip(axes, image_ids):
        image_path, mask_path = image_id_to_paths(image_id)
        image = load_grayscale_array(image_path)
        mask = load_binary_mask(mask_path)
        overlay = make_overlay(image, mask)
        ax.imshow(overlay)
        ax.set_title(f"ID {image_id}\narea={int(mask.sum())}")
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

## 12. Random Positive, Empty, Small, and Large Mask Examples

In [ ]:
positive_ids = mask_stats_df.loc[mask_stats_df["positive_pixels"] > 0, "imageID"].tolist()
empty_ids = mask_stats_df.loc[mask_stats_df["positive_pixels"] == 0, "imageID"].tolist()

random_positive_ids = random.sample(positive_ids, min(8, len(positive_ids)))
random_empty_ids = random.sample(empty_ids, min(8, len(empty_ids)))

small_positive_ids = mask_stats_df.loc[mask_stats_df["positive_pixels"] > 0].sort_values("positive_pixels").head(8)["imageID"].tolist()
large_positive_ids = mask_stats_df.sort_values("positive_pixels", ascending=False).head(8)["imageID"].tolist()

plot_overlay_grid(random_positive_ids, "Random Positive Mask Examples")
plot_overlay_grid(random_empty_ids, "Random Empty Mask Examples")
plot_overlay_grid(small_positive_ids, "Smallest Positive Masks")
plot_overlay_grid(large_positive_ids, "Largest Positive Masks")

## 13. Inspect Test Images

Test images do not have masks, but they should be checked for shape, mode, intensity range, and visual similarity to training images.

In [ ]:
def plot_test_grid(paths, title="Random Test Images"):
    selected = random.sample(paths, min(12, len(paths)))
    cols = 4
    rows = int(np.ceil(len(selected) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, path in zip(axes, selected):
        image = load_grayscale_array(path)
        ax.imshow(image, cmap="gray")
        ax.set_title(f"ID {path.stem}\nmean={image.mean():.1f}, std={image.std():.1f}")
        ax.axis("off")

    for ax in axes[len(selected):]:
        ax.axis("off")

    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


plot_test_grid(test_image_paths)

## 14. EDA Findings Template

Fill this in after running the notebook.

### File Alignment

- Train images found:
- Train masks found:
- Test images found:
- Aligned train image/mask pairs:
- Missing masks:
- Missing images:

### Image Format

- Image extensions observed:
- Mask extensions observed:
- Image modes observed:
- Mask modes observed:
- Image sizes observed:

### Mask Sparsity

- Empty masks:
- Positive masks:
- Median positive needle area percentage:
- Largest mask area:
- Smallest non-empty mask area:

### Intensity Findings

- Needle generally brighter than background?
- Any cases where the needle is not brighter?
- Train/test intensity distributions similar?

### Modeling Implications

- Recommended normalization:
- Recommended augmentations:
- Expected segmentation difficulty:
- Post-processing implications:

## 15. Immediate Next Steps After EDA

1. Confirm the image/mask ID alignment is complete.
2. Decide whether model code should load `.jpg`, `.png`, or all supported image extensions.
3. Use the mask area distribution to choose validation diagnostics for small needles.
4. Use the intensity analysis to decide whether classical thresholding is a useful baseline.
5. Build the PyTorch dataset class using ID-based pairing rather than filename equality.
6. Build the first U-Net baseline and validate with DICE, sensitivity, and the combined Kaggle metric.